# OCI-hosted reranking with OCI Generative AI and VecDB search

Run a semantic search using Oracle VecDB, then rerank the returned candidates with a reranking model hosted by OCI Generative AI. The vector search stays in VecDB; the reranking request is sent to the OCI Generative AI RerankText API.

Run Sections 1 through 6 in order. Section 7 is optional cleanup.

## Prerequisites

- Run Section 1 to install `oracle-vecdb`, `oci`, `python-dotenv`, and `pandas` in the notebook's Python environment.
- Copy `.env.example` to `.env` and fill in the VecDB connection and OCI configuration values.
- The OCI principal must be allowed to use Generative AI in the selected compartment.
- Use an OCI Generative AI region where the selected reranker is available on-demand. The default `cohere.rerank-v4.0-fast` uses `us-ashburn-1`; Phoenix currently exposes this model as dedicated-only. See the [regional availability table](https://docs.oracle.com/en-us/iaas/Content/generative-ai/model-endpoint-regions.htm).
- Have an embedding model available in VecDB. The default demo uses `ALL_MINILM_L12_V2`.

The default table is named `RERANK_SEARCH_DEMO`. It is safe to reuse for this notebook and can be removed in Section 7. This notebook does not require the in-database model-loading notebook.

## 1. Install notebook dependencies

Install the packages used by this notebook in the Python environment that runs Jupyter.

In [ ]:
%pip install --upgrade oracle-vecdb oci python-dotenv pandas

## 2. Configure the demo

Load the shared `.env` file and define the OCI reranker, search table, query, and result count. Change only these optional settings when adapting the example to another integrated-embedding table.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

ENV_FILE = Path(".env")
if not ENV_FILE.is_file():
    raise FileNotFoundError(
        "Copy .env.example to .env in the current directory first."
    )
load_dotenv(ENV_FILE, override=True)

OCI_RERANK_MODEL = os.getenv("OCI_RERANK_MODEL", "").strip() or "cohere.rerank-v4.0-fast"
OCI_GENAI_REGION = os.getenv("OCI_GENAI_REGION", "").strip() or "us-ashburn-1"
SEARCH_TABLE = os.getenv("RERANK_SEARCH_TABLE", "RERANK_SEARCH_DEMO").strip()
SEARCH_EMBEDDING_MODEL = os.getenv("RERANK_SEARCH_EMBEDDING_MODEL", "ALL_MINILM_L12_V2").strip()
SEARCH_TEXT_FIELD = os.getenv("RERANK_SEARCH_TEXT_FIELD", "TEXT").strip()
SEED_DEMO_TABLE = os.getenv("RERANK_SEARCH_SEED", "true").lower() == "true"
QUERY = os.getenv("RERANK_SEARCH_QUERY", "Which step should improve the order of passages retrieved by vector search before they are used to build an answer?")
TOP_K = int(os.getenv("RERANK_SEARCH_TOP_K", "6"))

required = ["VECDB_REST_URL", "OCI_CONFIG_FILE"]
missing = [name for name in required if not os.getenv(name)]
if not os.getenv("VECDB_ACCESS_TOKEN") and not (os.getenv("VECDB_USERNAME") and os.getenv("VECDB_PASSWORD")):
    missing.append("VECDB_ACCESS_TOKEN or VECDB_USERNAME/VECDB_PASSWORD")
if missing:
    raise ValueError("Set these values in .env: " + ", ".join(missing))

print(f"OCI reranker: {OCI_RERANK_MODEL}")
print(f"OCI Generative AI region: {OCI_GENAI_REGION}")
print(f"Search table: {SEARCH_TABLE}")
print(f"Query: {QUERY}")

## 3. Connect to VecDB and OCI Generative AI

Create a VecDB client for retrieval and an OCI Generative AI client for the external reranking call. A blank `OCI_COMPARTMENT_NAME` means the tenancy root compartment; otherwise the notebook resolves the exact child compartment name.

In [ ]:
import oci
from oracle_vecdb import Configuration, OracleVecDB
from oci.generative_ai_inference import GenerativeAiInferenceClient

if os.getenv("VECDB_ACCESS_TOKEN"):
    config = Configuration(
        rest_url=os.environ["VECDB_REST_URL"],
        access_token=os.environ["VECDB_ACCESS_TOKEN"],
    )
else:
    config = Configuration(
        rest_url=os.environ["VECDB_REST_URL"],
        username=os.environ["VECDB_USERNAME"],
        password=os.environ["VECDB_PASSWORD"],
    )
if os.getenv("VECDB_SELF_SIGNED_SSL", "false").lower() == "true":
    config.verify_ssl = False
vecdb = OracleVecDB(config)

oci_config = oci.config.from_file(
    file_location=os.path.expanduser(os.environ["OCI_CONFIG_FILE"]),
    profile_name=os.getenv("OCI_PROFILE", "DEFAULT"),
)
# RerankText requires a compartment OCID, so resolve a configured name.
compartment_name = os.getenv("OCI_COMPARTMENT_NAME", "").strip()
if compartment_name:
    compartments = oci.identity.IdentityClient(oci_config).list_compartments(
        oci_config["tenancy"], compartment_id_in_subtree=True,
        access_level="ACCESSIBLE", lifecycle_state="ACTIVE",
    ).data
    matches = [item for item in compartments if item.name == compartment_name]
    if len(matches) != 1:
        raise ValueError(f"Expected one accessible compartment named {compartment_name!r}; found {len(matches)}.")
    OCI_COMPARTMENT_ID = matches[0].id
else:
    # An empty name means the tenancy root compartment.
    OCI_COMPARTMENT_ID = oci_config["tenancy"]

genai_config = dict(oci_config)
genai_config["region"] = OCI_GENAI_REGION
genai = GenerativeAiInferenceClient(
    genai_config,
    service_endpoint=f"https://inference.generativeai.{OCI_GENAI_REGION}.oci.oraclecloud.com",
)
print("VecDB and OCI Generative AI clients are ready.")

## 4. Prepare a searchable document set

The default path creates an integrated-embedding table and upserts six small documents. VecDB creates the document embeddings using the hosted embedding model. If `RERANK_SEARCH_SEED=false`, the table must already exist and use integrated embeddings; the notebook will query its existing records without changing them.

In [ ]:
documents = [
    {"id": "doc-1", "metadata": {"TITLE": "Vector retrieval", "CATEGORY": "search", "TEXT": "Vector search retrieves a broad candidate set by comparing the query embedding with document embeddings."}},
    {"id": "doc-2", "metadata": {"TITLE": "Answer generation", "CATEGORY": "generative-ai", "TEXT": "After retrieval, a language model uses the passages in a prompt to generate an answer."}},
    {"id": "doc-3", "metadata": {"TITLE": "Cross-encoder reranking", "CATEGORY": "reranking", "TEXT": "A cross-encoder reranker scores each query-document pair and reorders the retrieved candidates before the application builds its prompt."}},
    {"id": "doc-4", "metadata": {"TITLE": "Embedding creation", "CATEGORY": "embeddings", "TEXT": "An embedding model converts each document into a vector before the document is stored in a vector table."}},
    {"id": "doc-5", "metadata": {"TITLE": "Prompt assembly", "CATEGORY": "generative-ai", "TEXT": "A RAG application combines a user question with retrieved evidence and sends that prompt to a generative model."}},
    {"id": "doc-6", "metadata": {"TITLE": "OCI Object Storage", "CATEGORY": "oci", "TEXT": "OCI Object Storage keeps model files and other artifacts in buckets."}},
]

# Reuse the table on reruns; create it only when it does not exist.
table_names = [item.table_name for item in (vecdb.list_vector_tables().items or [])]
if SEARCH_TABLE not in table_names:
    if not SEED_DEMO_TABLE:
        raise RuntimeError(f"Search table {SEARCH_TABLE!r} does not exist.")
    vecdb.create_vector_table(
        name=SEARCH_TABLE, comment="Small integrated-embedding corpus for OCI reranking demos",
        table_params={"auto_generate_id": False},
        embed_params={"model": SEARCH_EMBEDDING_MODEL, "embed_metadata_jsonpath": SEARCH_TEXT_FIELD},
        annotations={"TITLE": "string", "CATEGORY": "string", SEARCH_TEXT_FIELD: "string"},
    )
    print(f"Created integrated-embedding table: {SEARCH_TABLE}")
# Seed the six demo documents, or leave an existing table unchanged.
if SEED_DEMO_TABLE:
    vecdb.upsert_vectors(table_name=SEARCH_TABLE, vectors=documents)
    print(f"Upserted {len(documents)} demo documents.")
else:
    print(f"Using existing table without seeding: {SEARCH_TABLE}")

## 5. Search and rerank with OCI Generative AI

First ask VecDB for the initial candidate set and display it. Then pass those documents to OCI's `RerankText` API and display the reranked results. This is external reranking, so no reranker is loaded into VecDB. See the [OCI RerankText API](https://docs.oracle.com/en-us/iaas/tools/python/latest/api/generative_ai_inference/client/oci.generative_ai_inference.GenerativeAiInferenceClient.html).

In [ ]:
# Vector search: retrieve the initial candidate documents from VecDB.
search_response = vecdb.query(table_name=SEARCH_TABLE, query_by={"text": QUERY}, top_k=TOP_K, include_vectors=False)
search_items = search_response.items or []
if not search_items:
    raise RuntimeError("VecDB returned no search results.")

import pandas as pd
from IPython.display import display

initial_table = pd.DataFrame([
    {
        "Rank": rank,
        "Document": item.id,
        "Distance": item.distance,
        "Text": item.metadata.get(SEARCH_TEXT_FIELD, ""),
    }
    for rank, item in enumerate(search_items, 1)
])
initial_table["Distance"] = initial_table["Distance"].round(6)
print(f"Query: {QUERY}")
print("Initial vector distance search results:")
display(initial_table.style.set_properties(
    subset=["Text"],
    **{"width": "500px", "white-space": "normal", "vertical-align": "top"},
))

# OCI reranking: score those candidates and return them in relevance order.
candidate_documents = [item.metadata.get(SEARCH_TEXT_FIELD, "") for item in search_items]
if any(not text for text in candidate_documents):
    raise ValueError(f"Search results must contain text in metadata field {SEARCH_TEXT_FIELD!r}.")

request = oci.generative_ai_inference.models.RerankTextDetails(
    input=QUERY,
    compartment_id=OCI_COMPARTMENT_ID,
    serving_mode=oci.generative_ai_inference.models.OnDemandServingMode(model_id=OCI_RERANK_MODEL),
    documents=candidate_documents,
    top_n=len(candidate_documents),  # Return every candidate in ranked order.
    is_echo=True,
)
try:
    rerank_response = genai.rerank_text(request)
except oci.exceptions.ServiceError as exc:
    if exc.status == 404:
        raise RuntimeError(f"OCI could not find {OCI_RERANK_MODEL!r} in {OCI_GENAI_REGION!r}. Choose an on-demand region in .env or use a dedicated endpoint.") from exc
    raise

rerank_items = rerank_response.data.document_ranks or []
if not rerank_items:
    raise RuntimeError("OCI Generative AI returned no document ranks.")

reranked_table = []
for rank, item in enumerate(rerank_items, 1):
    index = int(item.index)  # Index into the original candidate list.
    source = search_items[index]
    reranked_table.append({
        "Rank": rank,
        "Document": source.id,
        "Score": float(item.relevance_score),
        "Text": source.metadata.get(SEARCH_TEXT_FIELD, ""),
    })
reranked_table = pd.DataFrame(reranked_table)
reranked_table["Score"] = reranked_table["Score"].round(6)
print(f"Query: {QUERY}")
print(f"Reranked results from OCI Generative AI model {OCI_RERANK_MODEL}:")
display(reranked_table.style.set_properties(
    subset=["Text"],
    **{"width": "500px", "white-space": "normal", "vertical-align": "top"},
))

## 6. What the two stages demonstrate

Vector search is the fast first-stage retrieval step: it produces a manageable candidate set. OCI Generative AI reranking is the precision step: the hosted cross-encoder evaluates the query against each candidate and returns the final order. In an application, use the reranked order when building the prompt or response.

## 7. Optional cleanup

Run this cell independently if you used the default `RERANK_SEARCH_DEMO` table and want to remove it. The cell reconnects to VecDB using `.env`. OCI Generative AI models are managed by Oracle and are not removed by this notebook.

In [ ]:
import os
from dotenv import load_dotenv
from oracle_vecdb import Configuration, OracleVecDB

load_dotenv(".env")
if os.getenv("VECDB_ACCESS_TOKEN"):
    config = Configuration(
        rest_url=os.environ["VECDB_REST_URL"],
        access_token=os.environ["VECDB_ACCESS_TOKEN"],
    )
else:
    config = Configuration(
        rest_url=os.environ["VECDB_REST_URL"],
        username=os.environ["VECDB_USERNAME"],
        password=os.environ["VECDB_PASSWORD"],
    )
if os.getenv("VECDB_SELF_SIGNED_SSL", "false").lower() == "true":
    config.verify_ssl = False
vecdb = OracleVecDB(config)
SEARCH_TABLE = os.getenv("RERANK_SEARCH_TABLE", "RERANK_SEARCH_DEMO").strip()

if SEARCH_TABLE == "RERANK_SEARCH_DEMO":
    table_names = [item.table_name for item in (vecdb.list_vector_tables().items or [])]
    if SEARCH_TABLE in table_names:
        vecdb.drop_vector_table(name=SEARCH_TABLE)
        print(f"Dropped demo table: {SEARCH_TABLE}")
    else:
        print(f"Demo table does not exist: {SEARCH_TABLE}")
else:
    print("Cleanup skipped because SEARCH_TABLE is not RERANK_SEARCH_DEMO.")